<a href="https://colab.research.google.com/github/AnIsAsPe/Filtrado__Colaborativo/blob/main/Notebooks/Entrenamiento_sistema_recomendacion_netflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Librerias

In [1]:
!pip install surprise

In [2]:
# Data Analysis
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Surprise: Simple Python RecommendatIon System Engine. Modelos de factorización matricial.
# https://surpriselib.com/
from surprise import Reader, Dataset, SVD
from surprise import dump, accuracy



### Carga de datos

Los datos probienen de la competencia de Netflix disponibles en Kaggle. (https://www.kaggle.com/datasets/netflix-inc/netflix-prize-data)

Para este ejercicio sólo cargaremos un archivo, pero la competencia tiene un total de 4 archivos.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df = pd.read_csv(
    "/content/drive/MyDrive/Datos/Netflix/combined_data_1.txt",
    usecols=[0,1],
    header=None,
    names=["CustomerID", "Rating"],
    low_memory=True,
)

In [5]:
df.head()

,CustomerID,Rating
0,1:,NaN
1,1488844,3.0
2,822109,5.0
3,885013,4.0
4,30878,4.0


### Analisis exploratorio

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24058263 entries, 0 to 24058262
Data columns (total 2 columns):
 #   Column      Dtype  
---  ------      -----  
 0   CustomerID  object 
 1   Rating      float64
dtypes: float64(1), object(1)
memory usage: 367.1+ MB


Veamos la presencia de datos nulos:

In [7]:
df.isna().sum()

,0
CustomerID,0
Rating,4499


Los registros que tenemos con valores nulos corresponden al Id de las películas.

In [9]:
# Identificar las filas que contienen el ID de la película
# Son aquellas donde el 'Rating' es NaN
filas_pelicula = df["Rating"].isna()
df[filas_pelicula]

,CustomerID,Rating
0,1:,NaN
548,2:,NaN
694,3:,NaN
2707,4:,NaN
2850,5:,NaN
...,...,...
24046714,4495:,NaN
24047329,4496:,NaN
24056849,4497:,NaN
24057564,4498:,NaN


In [10]:
# Creamos la columna y le asignamos el ID (quitando los dos puntos) solo en las filas correspondientes
df.loc[filas_pelicula, 'MovieID'] = df.loc[filas_pelicula, 'CustomerID'].str.replace(':', '')

# ffill() propaga el último valor no nulo hacia abajo, asignando el ID a todos los ratings de esa película
df['MovieID'] = df['MovieID'].ffill()


# Filtramos para eliminar las filas originales que solo tenían el ID de la película
df = df[~filas_pelicula].reset_index(drop=True)

#Optimización extrema de memoria (Esencial para archivos masivos)
# combined_data1.txt llega hasta el MovieID 4499, int16 soporta hasta 32,767 (int32 por seguridad si luego decidimos incluir los otros archivos)
df['MovieID'] = df['MovieID'].astype(np.int32)

# Los CustomerID tienen 7 dígitos, int32 soporta hasta 2 mil millones
df['CustomerID'] = df['CustomerID'].astype(np.int32)

# Los ratings son de 1 a 5, int8 es perfecto (soporta hasta 127)
df['Rating'] = df['Rating'].astype(np.int8)

# Reordenamos las columnas
df = df[['MovieID','CustomerID','Rating' ]]
df

,MovieID,CustomerID,Rating
0,1,1488844,3
1,1,822109,5
2,1,885013,4
3,1,30878,4
4,1,823519,3
...,...,...,...
24053759,4499,2591364,2
24053760,4499,1791000,2
24053761,4499,512536,5
24053762,4499,988963,3


In [12]:
movies = df["MovieID"].nunique()
print(f"Este es el número de películas que tenemos en este archivo: {movies}")

Este es el número de películas que tenemos en este archivo: 4499


In [13]:
reviews = df["Rating"].count()
print(f"Este es el número de calificaciones: {reviews}")

Este es el número de calificaciones: 24053764


In [14]:
users = df["CustomerID"].nunique()
print(f"Esta es la cantidad de usuarios que tenemos: {users}")

Esta es la cantidad de usuarios que tenemos: 470758


Ahora observemos como se distribuye la columna de rating.

In [11]:
df["Rating"].value_counts() / df["Rating"].count() * 100

,count
Rating,
4,33.615284
3,28.703121
5,22.892812
2,10.140089
1,4.648694


#### Reducción de Ruido

Reduciremos e:

1) Las películas que tiene pocas calificaciones.
2) Usuarios que han calificado muy pocas películas.



In [16]:
conteo_peliculas = df['MovieID'].value_counts()
conteo_peliculas

,count
MovieID,
1905,193941
2152,162597
3860,160454
4432,156183
571,154832
...,...
4294,44
915,43
3656,42


In [17]:

conteo_peliculas.describe(percentiles=[.05, .1, .2, .25, .5, .6,.7,.75])

,count
count,4499.000000
mean,5346.468993
std,16176.313851
min,36.000000
5%,96.900000
10%,116.000000
20%,161.000000
25%,192.000000
50%,552.000000
60%,907.800000


In [18]:
umbral_peliculas = conteo_peliculas.quantile(0.10)
print(f'Umbral: {umbral_peliculas}')

# Identificar las películas a conservar (las que tienen >= calificaciones que el umbral)
peliculas_a_conservar = conteo_peliculas[conteo_peliculas >= umbral_peliculas].index


# Filtrar el DataFrame original
df_trim = df[df["MovieID"].isin(peliculas_a_conservar)]
print(f'Películas conservadas: {len(peliculas_a_conservar)} de {movies}')

Umbral: 116.0
Películas conservadas: 4055 de 4499


Ahora observemos el caso del conteo para los usuarios.

In [19]:
conteo_usuarios = df_trim["CustomerID"].value_counts()
conteo_usuarios

,count
CustomerID,
305344,4025
387418,3991
2439493,3818
1664010,3630
2118461,3474
...,...
1917209,1
1775118,1
1557100,1


In [20]:
# Distribución de peliculas ranqueadas por usuario
conteo_usuarios.describe()

,count
count,470715.000000
mean,51.011485
std,73.898188
min,1.000000
25%,8.000000
50%,24.000000
75%,64.000000
max,4025.000000


Definimos el historial mínimo de cada usuario para que el algoritmo pueda encontrar patrones relevantes según los rankings que ha dado.

In [21]:
# Cuántos usuarios tiene menos de 20 calificaciones
percentil_20 = (conteo_usuarios < 20).mean() * 100
print(f"El  {percentil_20:.2f}% de los usuarios han calificado a menos de 20 películas.")

El  44.71% de los usuarios han calificado a menos de 20 películas.


In [22]:
umbral_usuarios= conteo_usuarios.quantile(0.4471)
print(f'Umbral: {umbral_usuarios}')

# Identificar las películas a conservar (las que tienen >= calificaciones que el umbral)
usuarios_a_conservar = conteo_usuarios[conteo_usuarios >= 20].index


# Filtrar el DataFrame original
df_trim = df_trim[df_trim['CustomerID'].isin(usuarios_a_conservar)]
print(f'Usuarios conservados: {len(usuarios_a_conservar)} de {users}')

Umbral: 20.0
Usuarios conservados: 260268 de 470758


In [23]:
df.shape, df_trim.shape

((24053764, 3), (22276439, 3))

In [24]:
df_trim.shape[0] / df.shape[0]

0.926110316871821

### Preparación de los datos

¿Cómo se vería la matriz si solo fueran 30 usuarios y 10 películas?

In [24]:
# Tomamos 30 usuarios y 10 películas muy populares
top_usuarios = df['CustomerID'].value_counts().sample(30).index
top_peliculas = df['MovieID'].value_counts().head(10).index

# Filtramos el DataFrame para quedarnos SOLO con las interacciones entre ellos
df_muestra = df[(df['CustomerID'].isin(top_usuarios)) & (df['MovieID'].isin(top_peliculas))]

# Creamos la matriz densa (Pivote) solo para este subconjunto
matriz_didactica = df_muestra.pivot(
    index='CustomerID',
    columns='MovieID',
    values='Rating'
)

matriz_visual = matriz_didactica.fillna("0")

print("--- Matriz Dispersa Didáctica (30 Usuarios x 10 Películas) ---")
print(matriz_visual)

--- Matriz Dispersa Didáctica (30 Usuarios x 10 Películas) ---
MovieID    571  1905 1962 2152 2452 3860 3938 3962 4306 4432
CustomerID                                                  
209733        0    0    0    0    0    0    0    0  5.0    0
247098        0  5.0    0    0    0    0    0    0    0    0
563397        0  2.0    0    0    0    0    0    0    0    0
681310        0    0    0  3.0    0    0    0    0    0    0
688702      4.0  3.0  4.0    0  4.0    0    0  4.0  4.0    0
827459      4.0  4.0  4.0  3.0  5.0  3.0  3.0    0  4.0  3.0
930452        0    0    0    0  4.0    0    0    0    0    0
959667      5.0    0    0  1.0    0    0    0    0  4.0    0
1017055       0    0    0    0    0    0  5.0    0  5.0    0
1076756     4.0    0    0  3.0    0    0    0    0    0  4.0
1319609     3.0  5.0  5.0    0  4.0  5.0    0  3.0  5.0    0
1532551       0    0    0  2.0    0    0    0    0    0    0
1564531       0  5.0  5.0    0    0    0  5.0  5.0    0  4.0
1571661       0  4.0  

La biblioteca surpise no necesita que transformamos el formato de los datos a una matriz, lo cual es computacionalmente muy conveniente, pero vamos a calcular las dimensiones que tendría la matriz.

In [25]:
# Obtenemos la cantidad de usuarios y películas únicas en df_trim
num_usuarios = df_trim['CustomerID'].nunique()
num_peliculas = df_trim['MovieID'].nunique()

# Calculamos las celdas totales teóricas y las celdas realmente ocupadas
total_celdas = num_usuarios * num_peliculas
celdas_ocupadas = len(df_trim) # Cada fila en el DataFrame es una calificación real

# Calculamos los porcentajes de densidad y vacío (Sparsity)
densidad = celdas_ocupadas / total_celdas
sparsity = 1 - densidad


print(f"Usuarios x Películas: {num_usuarios} x {num_peliculas}")
print(f"Total de celdas: {total_celdas:,}")
print(f"Calificaciones reales: {celdas_ocupadas:,}")
print(f"Densidad de la matriz: {densidad * 100:.4f}%")
print(f"Sparsity: {(sparsity) * 100:.4f}%")

Usuarios x Películas: 260268 x 4055
Total de celdas: 1,055,386,740
Calificaciones reales: 22,276,439
Densidad de la matriz: 2.1107%
Sparsity: 97.8893%


### Factorización de Matrices: Singular Value Decomposion.

SVD (Descomposición en Valores Singulares, por sus siglas en inglés) es una técnica de factorización de matrices que descompone una matriz en tres matrices más simples: una matriz de izquierda singular, una matriz diagonal singular y una matriz de derecha singular.

La descomposición en valores singulares es ampliamente utilizada en diversas áreas, incluyendo procesamiento de señales, procesamiento de imágenes, aprendizaje automático y **sistemas de recomendación**. En el contexto de sistemas de recomendación, la SVD se utiliza comúnmente para realizar factorización de matrices en el ámbito de la recomendación colaborativa.

La idea central detrás de la SVD es encontrar una representación latente de los datos, donde cada ítem y cada usuario están descritos por un conjunto de características (vectores de factores latentes). Al descomponer la matriz original en matrices de características, es posible aproximar la matriz original al multiplicar estas matrices de características.

En sistemas de recomendación, la SVD se utiliza para predecir las calificaciones de los usuarios para elementos que aún no han sido calificados, basándose en las características latentes aprendidas. Sin embargo, **es importante tener en cuenta que la SVD puede sufrir de problemas de escalabilidad y manejo de datos dispersos**, lo que ha llevado al desarrollo de variantes como SVD truncada y métodos de factorización de matrices más avanzados en sistemas de recomendación modernos.

In [25]:
%%time
print("1. Iniciando partición 80/20 por usuario (Esto puede tomar un par de minutos)...")
# Tomamos exactamente el 20% de las películas de cada usuario para el set de prueba
test_df = df_trim.groupby('CustomerID').sample(frac=0.2, random_state=42)

# El 80% restante se queda para el set de entrenamiento
train_df = df_trim.drop(test_df.index)

print(f"Datos de Entrenamiento: {len(train_df):,} registros")
print(f"Datos de Prueba: {len(test_df):,} registros")

1. Iniciando partición 80/20 por usuario (Esto puede tomar un par de minutos)...
Datos de Entrenamiento: 17,822,398 registros
Datos de Prueba: 4,454,041 registros
CPU times: user 23.7 s, sys: 2.19 s, total: 25.9 s
Wall time: 26.1 s


In [26]:
import psutil

# Función para monitorear la RAM disponible en Google Colab
def mostrar_ram(paso):
    ram_libre = psutil.virtual_memory().available / (1024.0 ** 3)
    print(f"[{paso}] RAM Libre estimada: {ram_libre:.2f} GB")

In [27]:
mostrar_ram("Despues de dividir train-test")

[Despues de dividir train-test] RAM Libre estimada: 8.48 GB


In [28]:
print("Guardando el 100% del Test en Drive (4.4 millones de filas)...")
test_df.to_csv('/content/drive/MyDrive/Datos/Netflix/test_completo.csv', index=False)

Guardando el 100% del Test en Drive (4.4 millones de filas)...


In [29]:
print("\n2. Formateando datos para la librería Surprise...")
reader = Reader(rating_scale=(1, 5))
data_train = Dataset.load_from_df(train_df[['CustomerID', 'MovieID', 'Rating']], reader)
trainset = data_train.build_full_trainset()


2. Formateando datos para la librería Surprise...


In [ ]:
%%time
import shutil


mostrar_ram("Inicio del proceso")

print("\n3. Entrenando el modelo SVD con todos los datos (Ve por un café, tomará tiempo)...")
# Usamos parámetros estándar. Reducimos n_epochs por la gran cantidad de info que tenemos.
svd_completo = SVD(n_epochs=15, lr_all=0.005, reg_all=0.02, random_state=42)
svd_completo.fit(trainset)

mostrar_ram("Post-Entrenamiento")


print("\n4. Guardando modelo (Guardado seguro en disco local primero)...")
ruta_local = '/content/modelo_svd_produccion.pkl'
ruta_drive = '/content/drive/MyDrive/Modelos/Netflix/modelo_svd_produccion.pkl'

# Guardamos en el almacenamiento temporal de Colab (ultra rápido y sin cortes de red)
from surprise import dump
dump.dump(ruta_local, algo=svd_completo)
print("Archivo creado localmente. Copiando a Google Drive...")

# Transferimos el archivo terminado a tu Drive
shutil.copy(ruta_local, ruta_drive)

print(f"¡Éxito total! Archivo guardado de forma segura en: {ruta_drive}")

[Inicio del proceso] RAM Libre estimada: 2.37 GB

3. Entrenando el modelo SVD con todos los datos (Ve por un café, tomará tiempo)...
[Post-Entrenamiento] RAM Libre estimada: 2.17 GB

4. Guardando modelo (Guardado seguro en disco local primero)...
Archivo creado localmente. Copiando a Google Drive...
¡Éxito total! Archivo guardado de forma segura en: /content/drive/MyDrive/Modelos/Netflix/modelo_svd_produccion.pkl
CPU times: user 4min 30s, sys: 4.5 s, total: 4min 34s
Wall time: 6min 31s


In [31]:

# Guardamos tambien el archivo sin ruido
df_trim.to_csv('/content/drive/MyDrive/Datos/Netflix/historial_limpio.csv', index=False)
